# Unified Preprocessing Pipeline

This notebook implements the unified preprocessing strategy defined in `PREPROCESSING_GUIDE.md`.

## Objectives:
1. **Initial Validation**: Shuffle, drop duplicates and nulls from the raw dataset.
2. **Structural Cleaning**: Remove GIFs, stickers, and tags. Drop tag-only comments.
3. **Text Normalization**: Collapse punctuation intensity (e.g., `!!!` -> `! `) and ensure spacing after all punctuation.
4. **Unified Emoji Mapping**: Map emojis to Intent and Topic tokens simultaneously.
5. **Language Filtering**: Keep only Arabic and Latin scripts.
6. **Full Dataset Export**: Save the cleaned dataset with Random IDs.

In [9]:
import pandas as pd
import re
import emoji
import random
import numpy as np
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42
INPUT_FILE = 'dataset.csv'
SAMPLE_SIZE = 1000 # Adjust for verbose speed

## 1. Initial Data Cleaning
Global shuffle, deduplication, and null-check.

In [10]:
print(f"Loading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
initial_count = len(df)

# 1. Drop absolute empty comments
df = df.dropna(subset=['comment_text'])
after_nulls = len(df)

# 2. Shuffle globally
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 3. Drop duplicates in comment_text
df = df.drop_duplicates(subset=['comment_text'])
after_dedup = len(df)

print(f"--- Initial Cleaning Results ---")
print(f"Raw rows: {initial_count}")
print(f"Dropped {initial_count - after_nulls} null rows.")
print(f"Global shuffle applied.")
print(f"Dropped {after_nulls - after_dedup} duplicate rows.")
print(f"Total unique usable rows: {len(df)}")

Loading dataset.csv...
--- Initial Cleaning Results ---
Raw rows: 43788
Dropped 82 null rows.
Global shuffle applied.
Dropped 9252 duplicate rows.
Total unique usable rows: 34454


## 2. Core Preprocessing Functions

In [11]:
def log_transform(step_name, original, result):
    if str(original).strip() != str(result).strip():
        print(f"  [{step_name}] ")
        print(f"    - Before: {original}")
        print(f"    - After : {result}")

def map_emojis(text):
    emojis_found = [e['emoji'] for e in emoji.emoji_list(text)]
    tokens = []
    
    # Intent Mappings (Expanded from extracted frequencies)
    appr = ['❤️', '💖', '💓', '🥰', '👏', '🌹', '💐', '💘', '😍', '👏🏼', '😁', '🔥', '👍', '♥️', '💪', '💯', '🤩', '😘', '☺️', '🤍', '😎', '😻', '🫡', '🫶', '🌷', '😄', '🌸', '🤗', '💋', '🌺', '👏🏻', '🫶🏾', '🫶🏽', '🎊', '🫶🏼', '🤎', '😀', '💛', '🩷', '👍🏻', '👍🏼', '👍🏾', '👍🏽', '👍🏿', '💝', '💞']
    comp = ['🤮', '😡', '👎', '💀', '💸', '😒', '😑', '😢', '❌', '🤦🏻‍♂️', '😱', '😔', '😞', '😩', '🤢', '☹️', '😠', '😖', '😰', '🤬', '📈', '☹', '😧', '🤦🏽', '🫤', '🙁', '🤦‍♂️', '😨', '🚩', '🤦🏻‍♀️', '👎🏻', '🤨']
    inq = ['❓', '❔', '🤔', '🧐', '📍', '📞', '🕒', '✍️', '✋']
    recom = ['👌', '🔝', '🌟', '✅', '🥇', '👑']
    
    if any(e in appr for e in emojis_found): tokens.append("[APPRECIATION]")
    if any(e in comp for e in emojis_found): tokens.append("[COMPLAINT]")
    if any(e in inq for e in emojis_found): tokens.append("[INQUIRY]")
    if any(e in recom for e in emojis_found): tokens.append("[RECOMMENDATION]")
    
    has_intent = bool(tokens)

    # Topic Mappings (Expanded from extracted frequencies)
    bouffe = ['🥘', '🍔', '🍕', '🥙', '🥗', '🍦','🎂', '🍨', '🍞', '🥖', '🍿', '🥤', '🥟', '🥕', '🍩', '😋', '🤤', '🍜', '🍣', '🥩', '🍰', '🥐', '🥪', '🌭', '🍟', '🦀', '🌮', '🦐', '🦞', '🥯', '🍝', '🍯', '🍓', '🍉', '🍒', '🍋', '🍎', '🥑', '🌯', '🍗', '🍖', '🍧', '🥧']
    price = ['💸', '💰', '💳', '💶', '💵', '💴', '📈', '🤑']
    treat = ['🧑‍🍳', '👨‍🍳', '👋', '🤝', '🫂', '👋🏻']
    srv = ['🕒', '⏳', '🍴', '🍽️']
    endroit = ['📍', '🧼', '🧹', '🤳']
    delivery = ['🛵', '🚚', '📦', '🚛']
    
    if any(e in bouffe for e in emojis_found): tokens.append("[BOUFFE]")
    if any(e in price for e in emojis_found): tokens.append("[PRICE]")
    if any(e in treat for e in emojis_found): tokens.append("[TREATMENT]")
    if any(e in srv for e in emojis_found): tokens.append("[SERVICE]")
    if any(e in endroit for e in emojis_found): tokens.append("[ENDROIT]")
    if any(e in delivery for e in emojis_found): tokens.append("[DELIVERY]")
    
    has_topic = len(tokens) > (1 if has_intent else 0)


    clean_text = emoji.replace_emoji(text, replace="")
    return (clean_text + " " + " ".join(tokens)).strip()

def full_pipeline(text, verbose=True):
    if not isinstance(text, str): return ""
    current = text.strip()
    if verbose: print(f"\n--- Processing: '{current}'")

    # 1. Structural Cleaning
    temp = re.sub(r"\[GIF\]|\[Sticker\]", "", current)
    temp = re.sub(r"Replying to @[\w.]+[: ]*", "", temp)
    if verbose: log_transform("Structural", current, temp)
    current = temp

    # 2. Advanced Tag & Flair Removal
    temp = re.sub(r"@[^\s@]*", "", current)
    temp = temp.strip()
    if not temp:
        if verbose: print("  [DROP] Reason: Tags only comment")
        return ""
    if verbose: log_transform("Tag & Flair Removal", current, temp)
    current = temp

    # 3. Punctuation Collapse & Spacing
    temp = re.sub(r"([!?.])\1+", r"\1 ", current)
    temp = re.sub(r"([.!?,])(?=[^\s])", r"\1 ", temp)
    temp = re.sub(r"\s+", " ", temp).strip()
    if verbose: log_transform("Punctuation", current, temp)
    current = temp

    # 4. Emoji Mapping (Unified)
    final = map_emojis(current)
    if verbose: log_transform("Emoji Mapping", current, final)
    
    return final

def is_target_language(text):
    if len(text.strip()) < 2: return False
    try:
        lang = detect(text)
        return lang in ['ar', 'fr', 'en']
    except:
        return False

def generate_random_ids(n):
    ids = set()
    while len(ids) < n:
        ids.add(random.randint(100000, 999999))
    return list(ids)

## 3. Test on a Sample (Verbose)

In [12]:
sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
sample_df['raw_text'] = sample_df['comment_text']

print(f"Executing Unified Pipeline (Sample)...\n")
sample_df['comment_text'] = sample_df['comment_text'].apply(lambda x: full_pipeline(x, verbose=True))

pd.set_option('display.max_colwidth', None)
display(sample_df[['raw_text', 'comment_text']])

Executing Unified Pipeline (Sample)...


--- Processing: 'la kabylie d’hier et d’aujourd’hui , une nostalgie du civisme et de la propreté perdu.....on dirait Kaboul.'
  [Punctuation] 
    - Before: la kabylie d’hier et d’aujourd’hui , une nostalgie du civisme et de la propreté perdu.....on dirait Kaboul.
    - After : la kabylie d’hier et d’aujourd’hui , une nostalgie du civisme et de la propreté perdu. on dirait Kaboul.

--- Processing: 'c'est des prix très raisonnables .'

--- Processing: '@Houdaa Yadra kima gotlek'
  [Tag & Flair Removal] 
    - Before: @Houdaa Yadra kima gotlek
    - After : Yadra kima gotlek

--- Processing: 'ديكور برك و ضيق بزاف تقول راك قاعد معاهم كامل والماكلة على ربي و les prix منحكوش'

--- Processing: 'je suis kabyle 🥰'
  [Emoji Mapping] 
    - Before: je suis kabyle 🥰
    - After : je suis kabyle  [APPRECIATION]

--- Processing: 'It's so good but u have to clean more especially the couch they are dirty honestly'

--- Processing: 'كاين في باتنة'

--- Processi

,raw_text,comment_text
0,"la kabylie d’hier et d’aujourd’hui , une nostalgie du civisme et de la propreté perdu.....on dirait Kaboul.","la kabylie d’hier et d’aujourd’hui , une nostalgie du civisme et de la propreté perdu. on dirait Kaboul."
1,c'est des prix très raisonnables .,c'est des prix très raisonnables .
2,@Houdaa Yadra kima gotlek,Yadra kima gotlek
3,ديكور برك و ضيق بزاف تقول راك قاعد معاهم كامل والماكلة على ربي و les prix منحكوش,ديكور برك و ضيق بزاف تقول راك قاعد معاهم كامل والماكلة على ربي و les prix منحكوش
4,je suis kabyle 🥰,je suis kabyle [APPRECIATION]
...,...,...
995,Tellement pas cher,Tellement pas cher
996,Moi je pense ça serait une bonne idée d’aller directement en algerie,Moi je pense ça serait une bonne idée d’aller directement en algerie
997,Pauvre de nous,Pauvre de nous
998,ليامات الزينة 😍❤,ليامات الزينة [APPRECIATION]


## 4. Tag Verification (Targeted Test)
Specifically audit how the pipeline handles comments with @mentions.

In [13]:
print("Filtering for comments containing '@'...")
tag_df = df[df['comment_text'].str.contains("@", na=False)].copy()
tag_df['raw_text'] = tag_df['comment_text']

print(f"\nProcessing {len(tag_df)} tagged comments...\n")
tag_df['comment_text'] = tag_df['comment_text'].apply(lambda x: full_pipeline(x, verbose=True))

print("\n--- Tag Removal Result Comparison ---")
display(tag_df[['raw_text', 'comment_text']])

Filtering for comments containing '@'...

Processing 1438 tagged comments...


--- Processing: '@tojaratojakhahlou'
  [DROP] Reason: Tags only comment

--- Processing: '@(B) @Yasmin Zr'
  [Tag & Flair Removal] 
    - Before: @(B) @Yasmin Zr
    - After : Zr

--- Processing: '@👏👏👏👏'
  [DROP] Reason: Tags only comment

--- Processing: '@imariou30'
  [DROP] Reason: Tags only comment

--- Processing: '@Drvetcharlotte0'
  [DROP] Reason: Tags only comment

--- Processing: '@la princesse 👑❤️🥹 😂😂'
  [Tag & Flair Removal] 
    - Before: @la princesse 👑❤️🥹 😂😂
    - After : princesse 👑❤️🥹 😂😂
  [Emoji Mapping] 
    - Before: princesse 👑❤️🥹 😂😂
    - After : princesse   [APPRECIATION] [RECOMMENDATION]

--- Processing: '@Zinou Benyou @shabelbaroudmoulcarabina @Cha3tota @Soso Lina @rizriz238 @Elle by Myriam'
  [Tag & Flair Removal] 
    - Before: @Zinou Benyou @shabelbaroudmoulcarabina @Cha3tota @Soso Lina @rizriz238 @Elle by Myriam
    - After : Benyou    Lina   by Myriam
  [Punctuation] 
    - Befor

,raw_text,comment_text
17,@tojaratojakhahlou,
18,@(B) @Yasmin Zr,Zr
21,@👏👏👏👏,
51,@imariou30,
53,@Drvetcharlotte0,
...,...,...
43412,@ah•_•med @PAẞ LOU @9⁹9 👀👀,LOU
43442,@Koki's dream house 👑 عرضة على شوا، ولا غير قهوة كافية؟!!,dream house عرضة على شوا، ولا غير قهوة كافية؟! [RECOMMENDATION]
43461,@🌺🦄 Divine 🦄🌺,Divine [APPRECIATION]
43469,@Ab Mirou M la lumière thabal 😂😂😂😂😂😂 5ir mal makla,Mirou M la lumière thabal 5ir mal makla


## 5. Process & Export Full Dataset

In [14]:
recap = {}
recap['1_Load'] = len(df)

print(f"Applying pipeline to FULL dataset ({len(df)} rows)...")
final_df = df.copy()

final_df['comment_text'] = final_df['comment_text'].apply(lambda x: full_pipeline(x, verbose=False))
recap['2_Cleaned'] = len(final_df)

print("Removing empty results...")
final_df = final_df[final_df['comment_text'].str.strip() != ""]
final_df = final_df.dropna(subset=['comment_text'])
recap['3_NonEmpty'] = len(final_df)

print("Enforcing script rules (Arabic/Latin)...")
final_df = final_df[final_df['comment_text'].apply(is_target_language)]
recap['4_Language'] = len(final_df)

print("Final deduplication and shuffle...")
final_df = final_df.drop_duplicates(subset=['comment_text'])
recap['5_FinalUnique'] = len(final_df)

print("Assigning Random IDs...")
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)
final_df.insert(0, 'final_id', generate_random_ids(len(final_df)))

output_file = 'dataset_preprocessed.csv'
final_df.to_csv(output_file, index=False)

print(f"\nSUCCESS! Saved {len(final_df)} high-quality rows to {output_file}")

Applying pipeline to FULL dataset (34454 rows)...
Removing empty results...
Enforcing script rules (Arabic/Latin)...
Final deduplication and shuffle...
Assigning Random IDs...

SUCCESS! Saved 24788 high-quality rows to dataset_preprocessed.csv


## 6. Preprocessing Recap

In [16]:
recap_df = pd.DataFrame.from_dict(recap, orient='index', columns=['Row Count'])
recap_df['Yield %'] = (recap_df['Row Count'] / recap['1_Load'] * 100).round(2)
recap_df['Dropped'] = recap_df['Row Count'].diff().fillna(0).astype(int) * -1

print(f"\n--- Preprocessing Recap ---")
display(recap_df)
print(f"\nTotal usable data preserved: {recap_df.iloc[-1]['Yield %']}% of unique raw input.")


--- Preprocessing Recap ---


,Row Count,Yield %,Dropped
1_Load,34454,100.00,0
2_Cleaned,34454,100.00,0
3_NonEmpty,33716,97.86,738
4_Language,25167,73.05,8549
5_FinalUnique,24788,71.95,379



Total usable data preserved: 71.95% of unique raw input.
